In [1]:
from cgra import *
from kernels import *
from scripts import sat_to_csv
import random


In [2]:
kernel_name = "benchmarks/compigra/yuxuan/relu_majo"
version = "_unroll"

In [3]:
# Global variables
CGRB_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr = 20000

In [4]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [5]:
# Data
def configMemory(data, data_size):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------
    # &Im   &Im    data_size_addr   &Im 
    # &Im   &Im    &Im              &Im

    data_size_addr = first_addr + data_size * 4

    config_vals_col0 = [first_addr, first_addr]
    config_vals_col1 = [first_addr, first_addr]
    config_vals_col2 = [data_size_addr, first_addr]
    config_vals_col3 = [first_addr, first_addr]
    
    addr_config_loads_col0 = 0
    kernel_add_memory_region(kernel_name, addr_config_loads_col0, config_vals_col0, version=version)
    addr_config_loads_col1 = addr_config_loads_col0 + len(config_vals_col0)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col1, config_vals_col1, version=version)
    addr_config_loads_col2 = addr_config_loads_col1 + len(config_vals_col1)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col2, config_vals_col2, version=version)
    addr_config_loads_col3 = addr_config_loads_col2 + len(config_vals_col2)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col3, config_vals_col3, version=version)
    # Load data
    kernel_add_memory_region(kernel_name, first_addr, data, version=version)
    kernel_add_memory_region(kernel_name, data_size_addr, [data_size], version=version)
    # Config data address for direct loads
    load_addrs = [addr_config_loads_col0, addr_config_loads_col1, addr_config_loads_col2, addr_config_loads_col3]
    return load_addrs

In [6]:
def runKernel(load_addrs, max_it=1000, printVal=1):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [7]:
def getResult(first_addr_C, end_addr_C, vlen):
    result = [0 for _ in range(vlen)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [8]:
def relu_cpu(image, vlen):
    expected_res = [0 for _ in range(vlen)]
    for i in range(vlen):
        if image[i] > 0:
            expected_res[i] = image[i]
        else:
            expected_res[i] = 0
    return expected_res

In [9]:
# Test dimensions
IMAGE_SIZE = 64*64
image = [random.randint(-20, 20) for _ in range(IMAGE_SIZE)]


load_addrs = configMemory(image, IMAGE_SIZE)

In [10]:
runKernel(load_addrs, max_it=200000, printVal=0)

Instr =  0 ( 0 )
[   0,    0, 36384,    0]    [NOP , NOP , LWD ROUT  arg1, NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , NOP , NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , NOP , NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , SADD R0  ZERO ZERO, NOP ]    
-------
Instr =  1 ( 1 )
[   0, 20000, 4096, 20000]    [NOP , LWD R0  arg0, LWI ROUT  ROUT, LWD R0  arg0]    
[20000,    0, 36384, 20000]    [LWD R0  arg0, NOP , LWD R0  arg0, LWD R0  arg0]    
[20000, 20000,    0,    0]    [LWD R0  arg0, LWD R0  arg0, NOP , NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , SADD ROUT  ZERO ZERO, NOP ]    
-------
Instr =  2 ( 2 )
[   0, 20000, 4096, 20000]    [NOP , NOP , NOP , NOP ]    
[20000,    0, 1024, 20000]    [NOP , NOP , SRA R1  RCT 2, NOP ]    
[20000, 20000,    0,    0]    [NOP , NOP , NOP , NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , NOP , NOP ]    
-------
Instr =  3 ( 3 )
[   0, 20000, 4096, 20000]    [NOP , NOP , NOP , NOP ]    
[20000,    0, 1024, 20000]    [NOP ,

In [11]:
# Get result from CGRA
end_addr = first_addr + IMAGE_SIZE*4
result = getResult(first_addr, end_addr, IMAGE_SIZE)

# Get cpu output
expected_res = relu_cpu(image, IMAGE_SIZE)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors))
    print("CGRA: " + str(result))
    print("CPU:  " + str(expected_res))
else:
    print("OK")



Err: 504
CGRA: [0, -1, 0, 13, 4, -13, 0, 0, 0, 20, 6, 0, 0, -6, 0, 2, 0, -6, 0, 18, 9, -18, 9, 0, 0, -11, 13, 4, 0, -8, 0, 4, 0, 18, 12, 19, 10, 6, 0, 13, 15, -17, 0, 0, 5, 4, 0, 0, 12, -3, 0, 3, 0, 7, 0, 0, 0, 17, 0, 0, 0, 5, 0, 0, 4, -4, 0, 0, 0, 19, 0, 9, 0, -16, 0, 0, 0, -10, 5, 19, 5, 14, 0, 0, 0, 8, 15, 0, 11, 20, 14, 18, 5, 3, 0, 0, 17, -18, 17, 0, 10, -7, 11, 3, 0, -5, 0, 0, 17, -2, 0, 0, 0, -15, 17, 10, 0, 10, 0, 0, 0, 7, 16, 0, 0, -16, 0, 7, 0, -5, 8, 0, 0, 20, 8, 0, 0, -2, 0, 0, 1, 10, 0, 15, 9, 11, 2, 0, 0, 3, 8, 19, 15, -16, 19, 0, 0, -3, 9, 0, 0, 16, 9, 0, 2, -11, 9, 16, 12, 6, 7, 0, 0, 0, 4, 0, 0, 18, 0, 17, 1, -3, 0, 2, 0, 16, 1, 0, 9, -11, 7, 16, 0, 16, 12, 11, 0, -18, 0, 19, 0, 1, 0, 16, 0, -12, 0, 0, 14, -15, 19, 0, 20, -4, 12, 4, 0, -11, 10, 0, 15, -16, 0, 13, 13, 11, 20, 0, 19, 4, 0, 10, 16, 1, 0, 0, 0, -2, 0, 12, 5, 12, 0, 0, 0, 18, 9, 18, 6, 4, 0, 0, 20, -6, 17, 0, 17, 4, 15, 12, 5, 20, 0, 12, 0, 14, 18, 0, 11, 14, 13, 0, 0, 11, 0, 0, 0, 6, 0, 0, 2, 7, 18, 14, 0,